In [55]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


**Data Collection**

In [4]:
df = pd.read_csv(r"/content/drive/MyDrive/Deep Learning/customer_sentimental_analysis/Customer_Sentiment.csv")
df

,customer_id,gender,age_group,region,product_category,purchase_channel,platform,customer_rating,review_text,sentiment,response_time_hours,issue_resolved,complaint_registered
0,1,male,60+,north,automobile,online,flipkart,1,very disappointed with the quality.,negative,46,yes,yes
1,2,other,46-60,central,books,online,swiggy instamart,5,fast delivery and great packaging.,positive,5,yes,no
2,3,female,36-45,east,sports,online,facebook marketplace,1,very disappointed with the quality.,negative,38,yes,yes
3,4,female,18-25,central,groceries,online,zepto,2,product stopped working after few days.,negative,16,yes,yes
4,5,female,18-25,east,electronics,online,croma,3,neutral about the quality.,neutral,15,yes,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,24996,female,36-45,south,beauty,online,lenskart,1,very disappointed with the quality.,negative,40,yes,yes
24996,24997,other,60+,central,automobile,online,flipkart,5,"amazing experience, highly recommend!",positive,25,yes,no
24997,24998,male,18-25,south,beauty,online,ajio,4,fast delivery and great packaging.,positive,9,yes,no
24998,24999,female,26-35,central,automobile,online,snapdeal,5,great value for money.,positive,65,no,no


**Data Understanding**

In [35]:
customer_review_sentiment = pd.DataFrame(data = {"review":df['review_text'],"sentiment":df['sentiment']})
customer_review_sentiment

,review,sentiment
0,very disappointed with the quality.,negative
1,fast delivery and great packaging.,positive
2,very disappointed with the quality.,negative
3,product stopped working after few days.,negative
4,neutral about the quality.,neutral
...,...,...
24995,very disappointed with the quality.,negative
24996,"amazing experience, highly recommend!",positive
24997,fast delivery and great packaging.,positive
24998,great value for money.,positive


**Data Preparations**

In [36]:
stopwords = stopwords.words('english')
lemmitizer = WordNetLemmatizer()

AttributeError: 'list' object has no attribute 'words'

In [37]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmitizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

In [43]:
customer_review_sentiment['review'] = customer_review_sentiment['review'].apply(preprocess)
customer_review_sentiment


,review,sentiment
0,disappointed quality,negative
1,fast delivery great packaging,positive
2,disappointed quality,negative
3,product stopped working day,negative
4,neutral quality,neutral
...,...,...
24995,disappointed quality,negative
24996,amazing experience highly recommend,positive
24997,fast delivery great packaging,positive
24998,great value money,positive


****Model Building****

In [60]:
count_vector=CountVectorizer()
x = count_vector.fit_transform(customer_review_sentiment['review'])
x_df = pd.DataFrame(x.toarray(),columns=count_vector.get_feature_names_out())
y = customer_review_sentiment['sentiment']
x_df



,amazing,average,better,could,customer,day,decent,delivery,disappointed,exceeded,...,recommend,satisfied,service,special,stopped,unhelpful,value,work,working,worth
0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
24996,1,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
24997,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
24998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [58]:

x_train,x_test,y_train,y_test = train_test_split(x_df,y,test_size=0.2,random_state=42,stratify=y,shuffle=True)

In [56]:
model = LogisticRegression()

**Model Training**

In [59]:
model.fit(x_train,y_train)

LogisticRegression()

**Model Testing**

In [62]:
y_pred = model.predict(x_test)
y_pred

array(['neutral', 'neutral', 'positive', ..., 'negative', 'positive',
       'positive'], dtype=object)

**Model Evaluation**

In [63]:
confusion_matrix(  y_test,y_pred)

array([[1987,    0,    0],
       [   0, 1017,    0],
       [   0,    0, 1996]])

In [65]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00      1987
     neutral       1.00      1.00      1.00      1017
    positive       1.00      1.00      1.00      1996

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000

